# Backtest Statistics

## Problem Definition

**Question.** What do the frozen primary-only and meta-filtered holdout results show after explicit costs?

**Role in the workflow.** Provide the main comparative backtest report, with meta-filtered as the primary strategy result.

**Inputs.** Final event return stream, primary/meta predictions, and CPCV paths.

**Outputs.** `data/backtest_results/backtest_statistics.parquet` plus holdout cumulative-return series.

**Why this method.** Performance, run, cost, efficiency, and classification metrics expose both economic and predictive behavior.

**Assumptions.** Irregular events are treated as bets; annualization uses the observed event frequency and no tuning follows this report.

**Handoff.** Holdout statistics to `strategy_risk.ipynb` and the final research interpretation.


## One-Time Holdout Return Review

Both benchmarks use identical observed returns and the same 5 bp entry plus 5 bp exit convention. The main backtest is meta-filtered; primary-only is retained to reveal whether filtering and sizing helped.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model_backtesting.backtest_statistics import (
    ClassificationScores,
    Efficiency,
    ImplementationShortfall,
    Performance,
    Runs,
)

result_dir = PROJECT_ROOT / "data/backtest_results"
returns = pd.read_parquet(result_dir / "event_strategy_returns.parquet").sort_index()
holdout = returns[returns["partition"].eq("holdout")].copy()
elapsed_years = (holdout.index.max() - holdout.index.min()).total_seconds() / (365.25 * 24 * 3600)
events_per_year = len(holdout) / elapsed_years

rows = []
for strategy in ["primary_only", "meta_filtered"]:
    net = holdout[f"{strategy}_net_return"]
    gross = holdout[f"{strategy}_gross_return"]
    cost = holdout[f"{strategy}_total_cost"]
    wealth = (1.0 + net).cumprod()
    drawdown = Runs.drawdown(wealth)
    rows.append(
        {
            "strategy": strategy,
            "events": len(net),
            "gross_return_sum": float(gross.sum()),
            "total_cost_sum": float(cost.sum()),
            "net_return_sum": float(net.sum()),
            "compound_net_return": float(wealth.iloc[-1] - 1.0),
            "hit_ratio": float(Performance.hit_ratio(net)),
            "average_hit": float(Performance.average_return_from_hits(net)),
            "average_miss": float(Performance.average_return_from_misses(net)),
            "annualized_sharpe": float(Efficiency.annualized_sharpe_ratio(net, periods_per_year=events_per_year)),
            "maximum_drawdown": float(drawdown.max()) if len(drawdown) else 0.0,
            "return_on_execution_costs": float(ImplementationShortfall.return_on_execution_costs(gross.sum(), cost.sum())),
        }
    )

statistics = pd.DataFrame(rows).set_index("strategy")
classification = pd.DataFrame(
    {
        "primary": {
            "accuracy": ClassificationScores.accuracy(holdout["direction_label"], holdout["primary_side"]),
            "precision": ClassificationScores.precision(holdout["direction_label"], holdout["primary_side"]),
            "recall": ClassificationScores.recall(holdout["direction_label"], holdout["primary_side"]),
            "f1": ClassificationScores.f1_score(holdout["direction_label"], holdout["primary_side"]),
        },
        "meta": {
            "accuracy": ClassificationScores.accuracy(holdout["meta_label"], holdout["meta_action"]),
            "precision": ClassificationScores.precision(holdout["meta_label"], holdout["meta_action"]),
            "recall": ClassificationScores.recall(holdout["meta_label"], holdout["meta_action"]),
            "f1": ClassificationScores.f1_score(holdout["meta_label"], holdout["meta_action"]),
        },
    }
)

statistics.to_parquet(result_dir / "backtest_statistics.parquet")
classification.to_parquet(result_dir / "classification_statistics.parquet")
pd.DataFrame(
    {
        "primary_only": (1.0 + holdout["primary_only_net_return"]).cumprod() - 1.0,
        "meta_filtered": (1.0 + holdout["meta_filtered_net_return"]).cumprod() - 1.0,
    }
).to_parquet(result_dir / "holdout_cumulative_returns.parquet")

display(statistics)
display(classification)


## Results, Limitations, and Handoff

A negative net return, weaker meta result, or cost-driven deterioration is reported without changing the pipeline. One AAPL year cannot establish regime robustness, capacity, live execution quality, or profitability.

The next notebook receives the final holdout economic and classification statistics. No conclusion in this notebook is evidence of live-trading profitability.
